# Data Transformation — Progress Report

Start from `train_clean` / `test_clean` produced by the existing preprocessing notebook. Reuse the baseline's five features and sklearn preprocessing structure. No model is trained and no dataset is exported.


In [1]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Support starting the kernel in either the repository root or scripts/.
SCRIPTS_DIR = Path.cwd() if (Path.cwd() / "preprocessing.ipynb").is_file() else Path.cwd() / "scripts"
PREPROCESSING_PATH = SCRIPTS_DIR / "preprocessing.ipynb"
assert PREPROCESSING_PATH.is_file(), "Run from the repository root or scripts directory."


## 1. Obtain the existing cleaned data

Execute the code cells of `preprocessing.ipynb` in a separate namespace in this kernel. Its working directory is temporarily set to `scripts/` so its existing `../data` paths work. This reuses the original cleaning rules without editing or copying them; a separately opened notebook does not share kernel variables automatically.


In [2]:
preprocessing_notebook = json.loads(PREPROCESSING_PATH.read_text(encoding="utf-8"))
preprocessing_namespace = {"__name__": "__main__"}
original_cwd = Path.cwd()
try:
    os.chdir(SCRIPTS_DIR)
    for cell in preprocessing_notebook["cells"]:
        if cell["cell_type"] == "code":
            source = "".join(cell["source"])
            exec(compile(source, str(PREPROCESSING_PATH), "exec"), preprocessing_namespace)
finally:
    os.chdir(original_cwd)

train_clean = preprocessing_namespace["train_clean"]
test_clean = preprocessing_namespace["test_clean"]


train (150000, 11) test (50000, 10)
columns: ['RENT_APPROVAL_DATE', 'TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL', 'FLOOR_AREA_SQM', 'FURNISHED', 'LEASE_COMMENCE_DATE', 'FEE', 'MONTHLY_RENT']
NaNs in train: 0
NaNs in test: 0
train raw: ['1 room', '1-room', '2 room', '2-room', '3 room', '3-room', '4 room', '4-room', '5 room', '5-room', 'executive']
test  raw: ['1-room', '2-room', '3-room', '4-room', '5-room', 'executive']
train after unify: ['1 room', '2 room', '3 room', '4 room', '5 room', 'executive']
test  after unify: ['1 room', '2 room', '3 room', '4 room', '5 room', 'executive']
train STREET: raw=1146, lowercase=576, case-only extras=570
test STREET: raw=1129, lowercase=573, case-only extras=556
train extra exact dups: 425
test  extra exact dups: 5508 (keep all rows)
FURNISHED train ['yes'] test ['yes']
FEE train [0.0] test [0.0]
train extra exact dups after label fixes: 641
train: (150000, 11) -> (149359, 9)
test: (50000, 10) -> (50000, 8)
FLAT_TYPE: ['1 room', '2 room', '

## 2. Construct the five baseline features

`month_index = (approval year − 2021) × 12 + approval month − 1`, so January 2021 is 0. `flat_age = approval year − lease commencement year` is an approximate age. Negative ages become missing values without deleting records or changing the cleaned inputs. Median filling happens later inside the modeling pipeline.

Keep `TOWN` and cleaned `FLAT_TYPE` labels as supplied by preprocessing. All other predictors are excluded.


In [3]:
TARGET = "MONTHLY_RENT"
NUM_COLS = ["floor_area_sqm", "flat_age", "month_index"]
CAT_COLS = ["TOWN", "FLAT_TYPE"]
FEATURE_COLS = NUM_COLS + CAT_COLS
REQUIRED_COLS = ["RENT_APPROVAL_DATE", "FLOOR_AREA_SQM", "LEASE_COMMENCE_DATE"] + CAT_COLS

for name, df in (("train_clean", train_clean), ("test_clean", test_clean)):
    assert set(REQUIRED_COLS).issubset(df.columns), f"Missing input columns in {name}"
assert TARGET in train_clean.columns


def make_features(df):
    X = pd.DataFrame(index=df.index)
    dates = pd.to_datetime(df["RENT_APPROVAL_DATE"], format="%Y-%m", errors="coerce")
    if dates.isna().any():
        raise ValueError("RENT_APPROVAL_DATE contains missing or invalid dates.")

    X["floor_area_sqm"] = pd.to_numeric(df["FLOOR_AREA_SQM"], errors="coerce")
    lease_year = pd.to_numeric(df["LEASE_COMMENCE_DATE"], errors="coerce")
    flat_age = dates.dt.year - lease_year
    X["flat_age"] = flat_age.mask(flat_age < 0)
    X["month_index"] = (dates.dt.year - 2021) * 12 + dates.dt.month - 1

    for col in CAT_COLS:
        X[col] = df[col].astype(object)
    return X.replace([np.inf, -np.inf], np.nan)


X = make_features(train_clean)
y = pd.to_numeric(train_clean[TARGET], errors="coerce")
X_test = make_features(test_clean)
assert y.notna().all() and np.isfinite(y).all(), "Target values must be finite."

print("X:", X.shape, "y:", y.shape, "X_test:", X_test.shape)
print(X.head().to_string())


X: (149359, 5) y: (149359,) X_test: (50000, 5)
   floor_area_sqm  flat_age  month_index        TOWN FLAT_TYPE
0            93.0       9.0            0     punggol    4 room
1           110.0      20.0            0    sengkang    5 room
2            65.0      45.0            0  queenstown    3 room
3            96.0      11.0            0    sengkang    4 room
4           102.0      20.0            0   sembawang    4 room


## 3. Define the unfitted modeling preprocessor

Reuse the baseline's median imputation → `StandardScaler` for numerical features and missing-category imputation → one-hot encoding (`drop="first"`, `handle_unknown="ignore"`) for categorical features.

Keep these learned transformations inside the modeling pipeline: split `X` / `y` first, then fit the pipeline only on the training subset. Validation and test use the fitted mappings and statistics. Scaling is retained for the existing linear baseline; tree models generally do not require it. This notebook does not call `fit` or `fit_transform`.


In [4]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="MISSING")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, NUM_COLS),
    ("categorical", categorical_pipeline, CAT_COLS),
])


## 4. Validate and hand off

Outputs are `X`, `y`, `X_test`, and the unfitted `preprocessor`. Training rows remain aligned with the cleaned target; test rows retain their original submission order. The modeling notebook should consume these objects instead of rebuilding features from raw CSVs, and attach `preprocessor` before its existing regressor.


In [5]:
assert list(X.columns) == list(X_test.columns) == FEATURE_COLS
assert len(X) == len(y) == len(train_clean)
assert len(X_test) == len(test_clean) == 50000
assert X.index.equals(y.index) and X.index.equals(train_clean.index)
assert X_test.index.equals(test_clean.index)
assert TARGET not in X.columns and TARGET not in X_test.columns

for name, cleaned, features in (("train", train_clean, X), ("test", test_clean, X_test)):
    dates = pd.to_datetime(cleaned["RENT_APPROVAL_DATE"], format="%Y-%m")
    raw_age = dates.dt.year - pd.to_numeric(cleaned["LEASE_COMMENCE_DATE"])
    negative_age = raw_age < 0
    assert features.loc[negative_age, "flat_age"].isna().all()
    assert features["flat_age"].dropna().ge(0).all()
    assert features["flat_age"].isna().sum() == negative_age.sum()
    assert features["month_index"].equals((dates.dt.year - 2021) * 12 + dates.dt.month - 1)
    pd.testing.assert_frame_equal(features[CAT_COLS], cleaned[CAT_COLS].astype(object))
    print(f"{name}: negative ages -> missing = {int(negative_age.sum())}; "
          f"month_index range = {features['month_index'].min()}..{features['month_index'].max()}")

assert not hasattr(preprocessor, "transformers_")
assert not hasattr(numeric_pipeline.named_steps["imputer"], "statistics_")
assert not hasattr(numeric_pipeline.named_steps["scaler"], "mean_")
assert not hasattr(categorical_pipeline.named_steps["onehot"], "categories_")
print("Checks passed: five features, aligned targets, all rows retained, cleaned labels preserved, preprocessor unfitted.")


train: negative ages -> missing = 1; month_index range = 0..50
test: negative ages -> missing = 0; month_index range = 50..66
Checks passed: five features, aligned targets, all rows retained, cleaned labels preserved, preprocessor unfitted.
